# Module 02 – Prompt Engineering & AI Productivity
**Level:** Beginner | **Duration:** Day 6 – Day 10

**Name:** _your name_  |  **Date:** _date_

## Key Objectives
1. Learn zero-shot and few-shot prompting, roles, contexts, instructions, and constraints.
2. Obtain structured outputs (such as JSON) and write prompts for summarization, classification, and generation.
3. Apply AI-assisted coding and debugging, and evaluate prompts using reusable templates and testing reports.
4. Build the **AI Productivity Toolkit** (reusable workflows for summarization, email, resume, coding/research assistance).

> **Run top to bottom.** Cells marked **✏️ Your turn** need your input. Section 7 is the deliverables checklist.
>
> **Model note:** This notebook works best with a hosted model (Claude) via an API key in Colab Secrets. Without a key it falls back to a small local model (`flan-t5-base`) – good enough to learn the mechanics, but weaker at JSON and long outputs.

## 0. Setup

In [1]:
!pip install -q anthropic transformers torch pandas tabulate

In [2]:
import os, re, json, time, textwrap
import pandas as pd
pd.set_option("display.max_colwidth", 100)

API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")   # Colab: 🔑 Secrets -> add ANTHROPIC_API_KEY, enable notebook access
except Exception:
    API_KEY = os.environ.get("ANTHROPIC_API_KEY")

MODEL = "claude-sonnet-5-5"
print("Using Claude ✅" if API_KEY else "No API key -> using local flan-t5-base fallback.")

No API key -> using local flan-t5-base fallback.


In [3]:
_flan = {}

def llm(prompt, system=None, temperature=0.3, max_tokens=600):
    """One function for every experiment. Claude if a key exists, else local flan-t5-base."""
    if API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=API_KEY)
        kwargs = dict(model=MODEL, max_tokens=max_tokens, temperature=temperature,
                      messages=[{"role": "user", "content": prompt}])
        if system:
            kwargs["system"] = system
        r = client.messages.create(**kwargs)
        return r.content[0].text.strip()
    # ---- local fallback (transformers v5-safe: no text2text pipeline) ----
    if not _flan:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        _flan["tok"] = AutoTokenizer.from_pretrained("google/flan-t5-base")
        _flan["model"] = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    full = f"{system}\n\n{prompt}" if system else prompt
    inputs = _flan["tok"](full, return_tensors="pt", truncation=True, max_length=1024)
    out = _flan["model"].generate(**inputs, max_new_tokens=min(max_tokens, 200))
    return _flan["tok"].decode(out[0], skip_special_tokens=True).strip()

print(llm("Say hello in one short sentence."))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Hello!


## 1. Prompting Fundamentals (Objective 1)

### 1.1 The anatomy of a good prompt
| Part | Purpose | Example |
|---|---|---|
| **Role** | Sets expertise and tone | "You are a senior technical editor." |
| **Context** | Background the model can't guess | "Audience: busy executives." |
| **Instruction** | The actual task (start with a verb) | "Summarize the text below." |
| **Constraints** | Length, format, style, what to avoid | "Max 3 bullets, no jargon." |
| **Examples** | Show the pattern (few-shot) | Input → Output pairs |
| **Input data** | The content to work on | Delimited text |

A helper that assembles these parts makes experiments repeatable.

In [4]:
def build_prompt(instruction, input_text=None, role=None, context=None,
                 constraints=None, examples=None, output_format=None):
    parts = []
    if role:        parts.append(f"You are {role}.")
    if context:     parts.append(f"Context: {context}")
    parts.append(f"Task: {instruction}")
    if constraints: parts.append("Constraints:\n" + "\n".join(f"- {c}" for c in constraints))
    if output_format: parts.append(f"Output format: {output_format}")
    if examples:
        ex = "\n\n".join(f"Input: {i}\nOutput: {o}" for i, o in examples)
        parts.append("Examples:\n" + ex)
    if input_text:  parts.append(f"Input:\n\"\"\"\n{input_text}\n\"\"\"")
    return "\n\n".join(parts)

article = ("Remote work has changed how teams collaborate. Companies report higher employee satisfaction "
           "and lower office costs, but managers struggle with onboarding, spontaneous communication, and "
           "measuring productivity. Hybrid models are emerging as a compromise, with fixed in-office days "
           "for collaboration and remote days for focused work.")
print(build_prompt("Summarize the text.", article, role="a business journalist",
                   constraints=["Exactly 2 sentences", "Neutral tone"]))

You are a business journalist.

Task: Summarize the text.

Constraints:
- Exactly 2 sentences
- Neutral tone

Input:
"""
Remote work has changed how teams collaborate. Companies report higher employee satisfaction and lower office costs, but managers struggle with onboarding, spontaneous communication, and measuring productivity. Hybrid models are emerging as a compromise, with fixed in-office days for collaboration and remote days for focused work.
"""


### 1.2 Zero-shot vs. few-shot

In [5]:
review = "The battery dies in two hours and the screen scratches easily. Not worth the price."

zero_shot = build_prompt("Classify the sentiment as Positive, Negative, or Neutral. Reply with one word.", review)

few_shot = build_prompt(
    "Classify the sentiment as Positive, Negative, or Neutral. Reply with one word.",
    review,
    examples=[("Absolutely love it, works perfectly!", "Positive"),
              ("It arrived on Tuesday in a brown box.", "Neutral"),
              ("Stopped working after a week. Terrible.", "Negative")])

print("ZERO-SHOT ->", llm(zero_shot, temperature=0))
print("FEW-SHOT  ->", llm(few_shot, temperature=0))

ZERO-SHOT -> Negative
FEW-SHOT  -> Output: Negative


**Rule of thumb:** start zero-shot. Add 2–5 diverse examples when the format is unusual, labels are subtle, or the model keeps drifting.

**✏️ Your turn:** Write a tricky review (sarcasm, mixed feelings) and compare zero-shot vs. few-shot.

In [6]:
tricky = "Oh great, another update that deletes my settings. Just what I needed."
print("ZERO:", llm(build_prompt("Classify the sentiment as Positive, Negative, or Neutral. One word.", tricky), temperature=0))
# TODO: add few-shot examples that teach sarcasm and compare

ZERO: Positive


### 1.3 Roles, context & constraints – same task, different prompts

In [7]:
variants = {
    "A_vague":       "Explain what an API is.",
    "B_audience":    build_prompt("Explain what an API is.", context="Audience: a 10-year-old."),
    "C_role+limits": build_prompt("Explain what an API is.", role="a friendly senior engineer mentoring an intern",
                                  constraints=["Use a restaurant analogy", "Max 60 words", "End with one follow-up question"]),
}
rows = []
for k, p in variants.items():
    out = llm(p, temperature=0.5)
    rows.append({"variant": k, "words": len(out.split()), "output": out})
pd.DataFrame(rows)

,variant,words,output
0,A_vague,17,APIs are software that allows a user to access a database without having to enter a password.
1,B_audience,12,APIs are software that allows users to modify and modify their applications.
2,C_role+limits,14,An API is a software that enables a user to interact with a computer.


**Prompting best practices checklist**
- ✅ Be specific: audience, goal, format, length
- ✅ Use delimiters (`"""`, XML tags, `###`) around input data
- ✅ Say what **to do**, not only what to avoid
- ✅ Give examples for tricky formats
- ✅ Ask for step-by-step reasoning on multi-step problems
- ✅ Iterate: change one thing at a time

## 2. Structured Outputs – JSON (Objective 2)

Programs need **predictable** output. Techniques:
1. Show the exact schema in the prompt.
2. Say "Return ONLY valid JSON, no prose, no code fences."
3. **Validate** in code; **retry/repair** on failure.

In [8]:
def extract_json(text):
    """Pull the first JSON object/array out of a model reply (handles ```json fences & chatter)."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    m = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if not m:
        raise ValueError("No JSON found")
    return json.loads(m.group(1))

def ask_json(prompt, required_keys=None, retries=2, **kw):
    last_err = None
    for attempt in range(retries + 1):
        p = prompt if attempt == 0 else (
            prompt + f"\n\nYour previous reply was invalid ({last_err}). Return ONLY valid JSON.")
        raw = llm(p, **kw)
        try:
            data = extract_json(raw)
            if required_keys:
                missing = [k for k in required_keys if k not in data]
                if missing:
                    raise ValueError(f"missing keys {missing}")
            return data
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"Failed after {retries+1} attempts: {last_err}\nLast raw reply: {raw[:200]}")

email_text = "Hi team, the client demo is moved to Friday 3pm. Priya please prep slides, Marco book the room. Budget cap is $2,000."
prompt = build_prompt(
    "Extract meeting details from the message.",
    email_text,
    output_format='Return ONLY valid JSON: {"event": str, "day": str, "time": str, "action_items": [{"owner": str, "task": str}], "budget": str|null}')
try:
    result = ask_json(prompt, required_keys=["event", "day", "time"], temperature=0)
    print(json.dumps(result, indent=2))
except RuntimeError as e:
    print("⚠️", e, "\n(Tip: small local models struggle with JSON – use an API key for best results.)")

⚠️ Failed after 3 attempts: No JSON found
Last raw reply: "event": str, "day": str, "time": str, "day": str, "event": str, "event": str, "budget": str|null" 
(Tip: small local models struggle with JSON – use an API key for best results.)


**✏️ Your turn:** Design a JSON schema to extract info from a job posting (title, company, skills[], salary_range, remote: bool) and test it.

In [9]:
job_posting = "Acme Corp is hiring a Senior Data Analyst (remote). You'll use SQL, Python and Tableau. Salary: $95k-$120k."
# TODO: write the prompt with build_prompt(...) and call ask_json(...)

## 3. Prompts for Summarization, Classification & Generation (Objective 2)

### 3.1 Summarization
Specify **audience, length, and format** (bullets vs. paragraph). Ask it to stay faithful to the source.

In [10]:
long_text = """The city council approved a $12 million plan to expand bike lanes by 40 miles over three years.
Supporters say it will cut emissions and traffic injuries. Opponents worry about reduced street parking
and construction disruption for small businesses. A pilot on Main Street reduced accidents by 18% last year.
Funding will come from a state transportation grant and a modest increase in parking fees."""

summaries = {
    "executive_1_line": build_prompt("Summarize for an executive.", long_text, constraints=["One sentence", "Include the dollar amount"]),
    "bullets":          build_prompt("Summarize as bullet points.", long_text, constraints=["3 bullets", "Only facts stated in the text"]),
    "pros_cons":        build_prompt("List arguments for and against.", long_text, output_format="Two sections: PROS, CONS (2 bullets each)"),
}
for k, p in summaries.items():
    print(f"--- {k} ---\n{llm(p, temperature=0.2)}\n")

--- executive_1_line ---
New York City has approved a plan to expand bike lanes in downtown Manhattan.

--- bullets ---
A city council vote in the city of San Francisco has approved a plan to expand bike lanes in the city's downtown area.

--- pros_cons ---
ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will be expanded to a new level. ProS: The bike lanes will



### 3.2 Classification
Give a **closed label set**, define each label, and demand a single label (or JSON with confidence).

In [11]:
tickets = [
    "I was charged twice for my subscription this month.",
    "How do I change my profile picture?",
    "The app crashes whenever I open the camera.",
    "Can you add a dark mode? It would be great.",
]
labels = {"Billing": "payment or invoice issues", "How-to": "usage questions", "Bug": "something is broken", "Feature": "new feature requests"}
label_desc = "\n".join(f"- {k}: {v}" for k, v in labels.items())

rows = []
for t in tickets:
    p = build_prompt(f"Classify the support ticket into exactly one label.\nLabels:\n{label_desc}\nReply with the label only.", t)
    rows.append({"ticket": t, "label": llm(p, temperature=0)})
pd.DataFrame(rows)

,ticket,label
0,I was charged twice for my subscription this month.,Billing
1,How do I change my profile picture?,How-to
2,The app crashes whenever I open the camera.,Bug
3,Can you add a dark mode? It would be great.,Feature


### 3.3 Generation
Provide purpose, audience, tone, length, and must-include points.

In [12]:
p = build_prompt("Write a product announcement.", role="a marketing copywriter",
    context="Product: 'FocusFlow', a desktop app that blocks distractions during deep work. Audience: remote software developers.",
    constraints=["Friendly, not hypey", "80-100 words", "Include one concrete benefit", "End with a call to action"])
print(llm(p, temperature=0.8, max_tokens=300))

FocusFlow is a desktop app that blocks distractions during deep work.


**✏️ Your turn:** Write one prompt for each task type (summarize / classify / generate) using your own topic. Note which constraint helped most.

In [13]:
my_summarize_prompt = ""
my_classify_prompt  = ""
my_generate_prompt  = ""
for name, pr in {"summarize": my_summarize_prompt, "classify": my_classify_prompt, "generate": my_generate_prompt}.items():
    if pr.strip():
        print(f"--- {name} ---\n{llm(pr)}\n")

## 4. AI-Assisted Coding & Debugging (Objective 3)

**Effective coding prompts include:** language/version, inputs & outputs, constraints (performance, style), edge cases, and the **full error message + code** when debugging.

> ⚠️ **Never trust AI code blindly.** Read it, run it, test it, and check for security issues (hard-coded secrets, injection, unsafe `eval`).

### 4.1 Generate code

In [14]:
gen_prompt = build_prompt(
    "Write a Python function `is_palindrome(s)` .",
    role="a careful senior Python developer",
    constraints=["Ignore case, spaces and punctuation", "Include type hints and a docstring",
                 "Handle empty string", "Return only a ```python code block"])
code_reply = llm(gen_prompt, temperature=0.2, max_tokens=500)
print(code_reply)

is_palindrome(s)


### 4.2 Test generated code – never skip this step

In [15]:
def extract_code(reply):
    m = re.search(r"```(?:python)?\n(.*?)```", reply, re.DOTALL)
    return m.group(1) if m else reply

namespace = {}
try:
    exec(extract_code(code_reply), namespace)      # only run code you've READ and understand
    f = namespace["is_palindrome"]
    tests = [("A man, a plan, a canal: Panama", True), ("hello", False), ("", True), ("No 'x' in Nixon", True)]
    for s, exp in tests:
        print("PASS" if f(s) == exp else "FAIL", repr(s))
except Exception as e:
    print("Generated code did not run:", e)

Generated code did not run: name 'is_palindrome' is not defined


### 4.3 Debug code
Give: **expected vs. actual behavior, the code, and the error**.

In [16]:
buggy_code = """
def average(nums):
    total = 0
    for i in range(1, len(nums)):
        total += nums[i]
    return total / len(nums)

print(average([10, 20, 30]))   # expected 20.0
"""
exec(buggy_code)   # prints 16.67 – wrong!

debug_prompt = build_prompt(
    "Find the bug, explain it in one sentence, then give the corrected function.",
    buggy_code,
    context="Expected output: 20.0 for [10, 20, 30]. Actual output: 16.666666666666668.",
    output_format="1) Bug explanation  2) Fixed code in a python block  3) One edge case to test")
print(llm(debug_prompt, temperature=0, max_tokens=500))

16.666666666666668
def average(nums): total = 0 for i in range(1, len(nums)): total += nums[i] return total / len(nums) print(average(nums, 10, 20, 30]) # expected 20.0


### 4.4 Other coding workflows

In [17]:
coding_prompt_templates = {
    "explain":   "Explain this code line by line for a beginner:\n```\n{code}\n```",
    "tests":     "Write pytest unit tests for this function, covering normal, edge and error cases:\n```\n{code}\n```",
    "refactor":  "Refactor for readability and performance without changing behavior. List what you changed:\n```\n{code}\n```",
    "review":    "Act as a code reviewer. List bugs, security issues, and style problems ranked by severity:\n```\n{code}\n```",
    "docstring": "Add Google-style docstrings and type hints. Return only code:\n```\n{code}\n```",
}
sample = "def f(x):\n    return [i*i for i in range(x) if i%2==0]"
print(llm(coding_prompt_templates["explain"].format(code=sample), temperature=0.2))

def f(x): return [i*i for i in range(x) if i%2==0]


**✏️ Your turn:** Pick a small function you wrote (or a deliberately broken one), then run *review* and *tests* prompts. Record what the AI got right and wrong.

## 5. Evaluating Prompts – Templates & Testing Reports (Objective 3)

Prompts are like code: version them, test them, measure them. The workflow:

1. Define a reusable **template** with variables.
2. Build a **test set** of inputs with expected results / checks.
3. **Run** every prompt version against every test.
4. Produce a **report** (pass rate, failures, latency) and iterate.

### 5.1 Reusable prompt template

In [18]:
class PromptTemplate:
    def __init__(self, name, template, version="1.0", system=None):
        self.name, self.template, self.version, self.system = name, template, version, system

    def render(self, **vars):
        missing = set(re.findall(r"{(\w+)}", self.template)) - set(vars)
        if missing:
            raise KeyError(f"Missing variables: {missing}")
        return self.template.format(**vars)

    def run(self, **vars):
        return llm(self.render(**vars), system=self.system, temperature=0)

sentiment_v1 = PromptTemplate("sentiment", "Classify sentiment of: {text}\nAnswer Positive, Negative or Neutral.", "1.0")
sentiment_v2 = PromptTemplate("sentiment",
    "You are a precise sentiment classifier.\nClassify the review below as exactly one of: Positive, Negative, Neutral.\n"
    "Sarcasm counts by its true meaning. Reply with ONE word only.\n\nReview: \"\"\"{text}\"\"\"", "2.0")
print(sentiment_v2.render(text="Great, it broke on day one."))

You are a precise sentiment classifier.
Classify the review below as exactly one of: Positive, Negative, Neutral.
Sarcasm counts by its true meaning. Reply with ONE word only.

Review: """Great, it broke on day one."""


### 5.2 Test set & automated checks

In [19]:
test_cases = [
    {"id": "T1", "text": "I love this phone, the camera is amazing!",           "expected": "Positive"},
    {"id": "T2", "text": "Worst purchase ever. Completely useless.",             "expected": "Negative"},
    {"id": "T3", "text": "The package contained a charger and a manual.",        "expected": "Neutral"},
    {"id": "T4", "text": "Oh wonderful, it died after one day. Fantastic.",      "expected": "Negative"},
    {"id": "T5", "text": "Not bad at all, exceeded my expectations.",            "expected": "Positive"},
    {"id": "T6", "text": "It's okay. Does what it says, nothing more.",          "expected": "Neutral"},
]

def check(output, expected):
    out = output.strip().strip(".").lower()
    return out == expected.lower()

def run_eval(template, cases):
    rows = []
    for c in cases:
        start = time.time()
        try:
            out = template.run(text=c["text"])
        except Exception as e:
            out = f"ERROR: {e}"
        rows.append({"prompt_version": template.version, "test_id": c["id"], "input": c["text"][:45],
                     "expected": c["expected"], "output": out[:40], "pass": check(out, c["expected"]),
                     "seconds": round(time.time() - start, 2)})
    return pd.DataFrame(rows)

report = pd.concat([run_eval(sentiment_v1, test_cases), run_eval(sentiment_v2, test_cases)], ignore_index=True)
report

,prompt_version,test_id,input,expected,output,pass,seconds
0,1.0,T1,"I love this phone, the camera is amazing!",Positive,Positive,True,0.31
1,1.0,T2,Worst purchase ever. Completely useless.,Negative,Negative,True,0.35
2,1.0,T3,The package contained a charger and a manual.,Neutral,Positive,False,0.29
3,1.0,T4,"Oh wonderful, it died after one day. Fantasti",Negative,Neutral,False,0.38
4,1.0,T5,"Not bad at all, exceeded my expectations.",Positive,Positive,True,0.30
5,1.0,T6,"It's okay. Does what it says, nothing more.",Neutral,Positive,False,0.31
6,2.0,T1,"I love this phone, the camera is amazing!",Positive,Positive,True,0.48
7,2.0,T2,Worst purchase ever. Completely useless.,Negative,Negative,True,0.52
8,2.0,T3,The package contained a charger and a manual.,Neutral,Neutral,True,0.56
9,2.0,T4,"Oh wonderful, it died after one day. Fantasti",Negative,Positive,False,0.47


### 5.3 Testing report

In [20]:
summary = (report.groupby("prompt_version")
           .agg(tests=("pass", "size"), passed=("pass", "sum"), avg_seconds=("seconds", "mean"))
           .assign(pass_rate=lambda d: (d.passed / d.tests * 100).round(1)))
display(summary)

print("\nFailures:")
display(report[~report["pass"]][["prompt_version", "test_id", "input", "expected", "output"]])

# Save a shareable report
report.to_csv("prompt_test_report.csv", index=False)
with open("prompt_test_report.md", "w") as f:
    f.write("# Prompt Test Report\n\n" + summary.reset_index().to_markdown(index=False) +
            "\n\n## Failures\n\n" + report[~report["pass"]].to_markdown(index=False))
print("\nSaved prompt_test_report.csv and prompt_test_report.md")

,tests,passed,avg_seconds,pass_rate
prompt_version,,,,
1.0,6,3,0.323333,50.0
2.0,6,5,0.505000,83.3



Failures:


,prompt_version,test_id,input,expected,output
2,1.0,T3,The package contained a charger and a manual.,Neutral,Positive
3,1.0,T4,"Oh wonderful, it died after one day. Fantasti",Negative,Neutral
5,1.0,T6,"It's okay. Does what it says, nothing more.",Neutral,Positive
9,2.0,T4,"Oh wonderful, it died after one day. Fantasti",Negative,Positive



Saved prompt_test_report.csv and prompt_test_report.md


**✏️ Your turn:** Create `sentiment_v3` (or a new template for a different task), add 3 more test cases (include edge cases), re-run, and explain in 3–4 sentences what improved and what still fails.

> _Your analysis:_

**Other useful checks:** JSON validity, required keys present, word-count limits, banned phrases absent, contains required facts, or an **LLM-as-judge** prompt scoring outputs 1–5 against a rubric.

## 6. Build the AI Productivity Toolkit (Objective 4)

A reusable module with workflows for:
- 📝 **Summarization** (text → bullets / executive summary)
- ✉️ **Email** (draft, reply, polish)
- 📄 **Resume** (tailor to a job description, improve bullets)
- 💻 **Coding assistant** (explain, debug, test, review)
- 🔎 **Research assistant** (question → structured research plan & notes)

The toolkit takes any `llm_fn`, so it works with Claude, local models, or future providers. We write it to a file (`ai_toolkit.py`) so you can import it anywhere.

In [21]:
%%writefile ai_toolkit.py
"""AI Productivity Toolkit – reusable prompt workflows."""
import json, re

class AIToolkit:
    def __init__(self, llm_fn):
        self.llm = llm_fn
        self.history = []          # simple usage log

    # ---------- helpers ----------
    def _run(self, workflow, prompt, system=None, temperature=0.3, max_tokens=700):
        out = self.llm(prompt, system=system, temperature=temperature, max_tokens=max_tokens)
        self.history.append({"workflow": workflow, "prompt_chars": len(prompt), "output_chars": len(out)})
        return out

    @staticmethod
    def _delim(label, text):
        return f"<{label}>\n{text}\n</{label}>"

    # ---------- 1. summarization ----------
    def summarize(self, text, style="bullets", max_items=5, audience="general reader"):
        styles = {
            "bullets":   f"Summarize as at most {max_items} bullet points.",
            "executive": "Write a 2-sentence executive summary.",
            "tldr":      "Write a one-sentence TL;DR.",
            "actions":   "Extract action items as a checklist with owners and deadlines if stated.",
        }
        p = (f"{styles[style]} Audience: {audience}. Use only information in the text; do not invent facts.\n\n"
             + self._delim("text", text))
        return self._run("summarize", p, temperature=0.2)

    # ---------- 2. email ----------
    def draft_email(self, purpose, recipient="a colleague", tone="professional and friendly",
                    key_points=None, max_words=150):
        pts = "\n".join(f"- {k}" for k in (key_points or []))
        p = (f"Write an email.\nRecipient: {recipient}\nPurpose: {purpose}\nTone: {tone}\n"
             f"Must include:\n{pts}\nConstraints: max {max_words} words; include a subject line; clear call to action; "
             f"no placeholders other than [Your Name].")
        return self._run("email", p, temperature=0.6)

    def reply_email(self, received_email, intent, tone="polite and concise"):
        p = (f"Draft a reply. My intent: {intent}. Tone: {tone}. Max 120 words.\n\n"
             + self._delim("received_email", received_email))
        return self._run("email_reply", p, temperature=0.5)

    def polish_text(self, text, goal="clearer and more concise"):
        p = f"Rewrite to be {goal}. Keep the meaning and the author's voice. Return only the rewritten text.\n\n" + self._delim("text", text)
        return self._run("polish", p, temperature=0.3)

    # ---------- 3. resume ----------
    def tailor_resume_bullets(self, resume_bullets, job_description):
        p = ("Rewrite the resume bullets to match the job description. Rules: keep every claim truthful (do NOT invent "
             "experience or numbers), start with strong action verbs, emphasize relevant keywords, max 25 words each.\n\n"
             + self._delim("job_description", job_description) + "\n"
             + self._delim("resume_bullets", resume_bullets))
        return self._run("resume_tailor", p, temperature=0.4)

    def resume_gap_analysis(self, resume_text, job_description):
        p = ("Compare the resume with the job description. Return ONLY JSON: "
             '{"match_score_0_100": int, "matching_skills": [str], "missing_skills": [str], "suggestions": [str]}\n\n'
             + self._delim("job_description", job_description) + "\n" + self._delim("resume", resume_text))
        raw = self._run("resume_gap", p, temperature=0)
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {"raw": raw}

    # ---------- 4. coding assistant ----------
    def code_assist(self, code, task="explain", extra=""):
        tasks = {
            "explain": "Explain this code step by step for a beginner.",
            "debug":   "Find bugs. For each: line, why it's wrong, fix. Then give corrected code.",
            "tests":   "Write pytest tests covering normal, edge and error cases.",
            "review":  "Review for bugs, security issues, readability. Rank by severity.",
            "refactor":"Refactor for readability without changing behavior; list changes.",
        }
        p = f"{tasks[task]} {extra}\n\n```\n{code}\n```"
        return self._run(f"code_{task}", p, temperature=0.2, max_tokens=900)

    # ---------- 5. research assistant ----------
    def research_plan(self, question, depth="beginner"):
        p = (f"I'm researching: {question}\nLevel: {depth}.\nProduce: 1) key sub-questions (5), 2) what to search for "
             f"(keywords), 3) types of credible sources, 4) common misconceptions, 5) a 5-step plan. "
             f"Do NOT fabricate citations or URLs; say if something needs verification.")
        return self._run("research_plan", p, temperature=0.4, max_tokens=900)

    def compare_options(self, options, criteria):
        p = (f"Compare these options: {', '.join(options)}\nCriteria: {', '.join(criteria)}\n"
             f"Return a markdown table, then a 2-sentence recommendation with caveats. Flag uncertain claims.")
        return self._run("compare", p, temperature=0.3, max_tokens=900)

    # ---------- utilities ----------
    def usage(self):
        import collections
        return dict(collections.Counter(h["workflow"] for h in self.history))

Writing ai_toolkit.py


### 6.1 Use the toolkit

In [22]:
import importlib, ai_toolkit
importlib.reload(ai_toolkit)
tk = ai_toolkit.AIToolkit(llm)

print("📝 SUMMARY\n", tk.summarize(long_text, style="bullets", max_items=3, audience="busy residents"), "\n")
print("✉️ EMAIL\n", tk.draft_email("Ask my manager for a one-week extension on the Q3 report",
      recipient="my manager Dana", key_points=["Data from finance arrived late", "New deadline: next Friday"]), "\n")

📝 SUMMARY
 A city council approved a plan to expand bike lanes in downtown San Francisco. 

✉️ EMAIL
 Please call me to discuss the Q3 report. I'm looking for a one-week extension. 



In [23]:
jd = "We need a Data Analyst with SQL, Python, dashboards (Tableau), and strong communication skills."
bullets = "- Made weekly sales reports in Excel\n- Helped the team with data questions\n- Built a dashboard in Power BI for managers"
print("📄 RESUME BULLETS\n", tk.tailor_resume_bullets(bullets, jd), "\n")
print("💻 CODE REVIEW\n", tk.code_assist("def add(a,b): return a-b", task="debug"), "\n")
print("🔎 RESEARCH PLAN\n", tk.research_plan("Does intermittent fasting improve focus?"), "\n")
print("Usage log:", tk.usage())

📄 RESUME BULLETS
 - Developed a dashboard in Power BI for managers 

💻 CODE REVIEW
 def add(a,b): return a-b 

🔎 RESEARCH PLAN
 Does intermittent fasting improve focus? 

Usage log: {'summarize': 1, 'email': 1, 'resume_tailor': 1, 'code_debug': 1, 'research_plan': 1}


### 6.2 Add your own workflow
**✏️ Your turn:** Add **one new workflow** to the toolkit (ideas: meeting notes → action items, study-notes → quiz, social post generator, SQL helper, translation with tone). Steps:
1. Write a prompt template with clear role, constraints, and delimiters.
2. Add it as a method (subclass below or edit `ai_toolkit.py`).
3. Test it with 3 inputs and add a test case to your evaluation harness from Section 5.

In [24]:
class MyToolkit(ai_toolkit.AIToolkit):
    def my_workflow(self, text):
        # TODO: replace with your own workflow
        p = "TODO: your prompt\n\n" + self._delim("text", text)
        return self._run("my_workflow", p)

# mytk = MyToolkit(llm)
# print(mytk.my_workflow("sample input"))

### 6.3 Download your toolkit & reports

In [25]:
try:
    from google.colab import files
    for f in ["ai_toolkit.py", "prompt_test_report.csv", "prompt_test_report.md"]:
        if os.path.exists(f):
            files.download(f)
except Exception:
    print("Not in Colab – files are saved in the working directory.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Deliverables Checklist

- [ ] Notebook runs end-to-end (Runtime ▸ Restart and run all)
- [ ] Section 1: zero-shot vs. few-shot comparison + role/context/constraint experiment
- [ ] Section 2: JSON extraction with validation (job-posting schema completed)
- [ ] Section 3: summarize / classify / generate prompts written for your own topic
- [ ] Section 4: coding + debugging prompts tested on real code, results noted
- [ ] Section 5: reusable template, test set (9+ cases), testing report, written analysis
- [ ] Section 6: AI Productivity Toolkit (`ai_toolkit.py`) with **your own added workflow**
- [ ] No API keys visible in code or outputs
- [ ] Shared: **File ▸ Save a copy in Drive**, then **Share ▸ Anyone with the link can view**

### Reflection (5–7 sentences)
1. Which prompting technique gave the biggest quality jump?
2. Where did the AI fail, and how did your tests catch it?
3. How would you use the toolkit in your daily work or studies?
4. What risks (accuracy, privacy, over-reliance) must you manage?

> _Your answer:_

### Summary
- Good prompts = **role + context + instruction + constraints + examples + format**.
- Use **few-shot** for tricky patterns; **JSON + validation + retry** for machine-readable output.
- Treat AI code as a draft: **run it, test it, review it**.
- **Evaluate prompts like software**: templates, test sets, pass-rate reports, versions.
- Reusable workflows turn one-off prompts into a **productivity toolkit**.

**Next:** Module 03.